# TP3: Recomendador de Disciplinas com NLP e Busca Semântica

**Fundamentos de Inteligência Artificial · Engenharia de Sistemas (UFMG)**

---

## Pipeline

1. **Curadoria:** 76 disciplinas com ementas, dificuldade e grade horária
2. **Embeddings (SBERT):** `neuralmind/bert-base-portuguese-cased` (768-d)
3. **Busca Semântica:** similaridade de cosseno entre perfil e ementas
4. **Filtros Estruturais:** pré-requisitos → dificuldade → horários
5. **Classificador BERT:** fine-tuning progressivo em 3 estágios


In [50]:
!pip install -q pandas numpy torch sentence-transformers scikit-learn transformers

import os, requests

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/AndreRezende1999/tp3-ufmg-recomendador/main/data"
DATA_FILE = "disciplinas_engsis.json"

if not os.path.exists(DATA_FILE):
    url = f"{GITHUB_RAW_BASE}/{DATA_FILE}"
    print(f"Baixando dados de {url}...")
    response = requests.get(url)
    response.raise_for_status()
    with open(DATA_FILE, "wb") as f:
        f.write(response.content)
    print("Dados baixados com sucesso.")
else:
    print("Dados já existem.")


Dados já existem.


In [51]:
import sys, os, json
import pandas as pd
import numpy as np
import torch
import warnings
from pathlib import Path
from sentence_transformers import SentenceTransformer, util
from dataclasses import dataclass
from typing import Iterable
from collections import Counter
warnings.filterwarnings('ignore')

MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
RANDOM_STATE = 42
DATA_FILE = "disciplinas_engsis.json"

print("Dependências carregadas.")


Dependências carregadas.


---
## Fase 1: Curadoria e Metodologia

Base de dados com 76 disciplinas de Engenharia de Sistemas (UFMG), integrando quatro fontes: ementas do site GEES UFMG (52 detalhadas, 24 inferidas), grade horária oficial (41), planilha colaborativa de dificuldade (50) e pré-requisitos do PPC (31).

As funções abaixo implementam a arquitetura híbrida do recomendador: curadoria dos textos, encoder SBERT com BERTimbau (768-d) e os três filtros estruturais determinísticos. Sem o SBERT, o recomendador seria uma simples verificação de regras; sem os filtros, as sugestões seriam coerentes mas academicamente inviáveis.


In [52]:
# 1.1 Curadoria dos dados

def load_disciplines():
    with open(DATA_FILE, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return pd.DataFrame(data)

def get_discipline_table():
    return load_disciplines()

def get_discipline_text(row):
    nome = row.get("nome", "")
    area = row.get("area_conhecimento", "")
    ementa = row.get("ementa", "")
    return f"{nome}. \u00c1rea: {area}. {ementa}"

def get_completed_profile_text(df, completed_codes):
    text_parts = []
    for code in completed_codes:
        disc = df[df["codigo"] == code]
        if not disc.empty:
            nome = disc.iloc[0]["nome"]
            area = disc.iloc[0]["area_conhecimento"]
            text_parts.append(f"{nome} ({area})")
    return "Disciplinas cursadas com interesse: " + ", ".join(text_parts) + "."

# Data curation functions loaded.


In [53]:
# 1.2 Encoder SBERT (BERTimbau, 768-d)

def load_sbert_model(model_name=MODEL_NAME):
    return SentenceTransformer(model_name)

def encode_texts(model, texts, batch_size=32):
    embeddings = model.encode(texts, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True)
    return embeddings

def encode_disciplines(model, df):
    texts = df.apply(get_discipline_text, axis=1).tolist()
    return encode_texts(model, texts)

def semantic_search(query_embedding, corpus_embeddings, top_k=10):
    query_tensor = torch.tensor(query_embedding)
    corpus_tensor = torch.tensor(corpus_embeddings)
    hits = util.semantic_search(query_tensor, corpus_tensor, top_k=top_k)[0]
    for hit in hits:
        hit['index'] = hit.pop('corpus_id')
    return hits

# SBERT encoder functions loaded.


In [54]:
# 1.3 Filtros estruturais (pré-requisitos, dificuldade, horários)

def prerequisites_satisfied(candidate_row, completed_codes):
    prerequisites = set(candidate_row.get("pre_requisitos", []) or [])
    return prerequisites.issubset(set(completed_codes))

def filter_recommended_candidates(candidates, completed_codes):
    if candidates.empty:
        return candidates.copy()
    mask = candidates.apply(lambda row: prerequisites_satisfied(row, completed_codes), axis=1)
    return candidates.loc[mask].copy()

def get_schedule_slots(row):
    horarios = row.get("horarios")
    if not isinstance(horarios, list) or not horarios:
        return []
    return [(h.get("dia", ""), h.get("turno", "")) for h in horarios if h.get("dia") and h.get("turno")]

def has_schedule_conflict(candidate_slots, selected_slots):
    if not candidate_slots:
        return False
    return any(slot in selected_slots for slot in candidate_slots)

def filter_by_schedule(candidates, already_selected_slots=None):
    if candidates.empty:
        return candidates.copy()
    if already_selected_slots is None:
        already_selected_slots = set()
    if not already_selected_slots:
        return candidates.copy()
    def _no_conflict(row):
        slots = get_schedule_slots(row)
        return not slots or not has_schedule_conflict(slots, already_selected_slots)
    mask = candidates.apply(_no_conflict, axis=1)
    return candidates.loc[mask].copy()

def greedy_balance_by_difficulty(candidates, *, difficulty_budget, max_courses=None, respect_schedule=True):
    if candidates.empty:
        return candidates.copy()
    ordered = candidates.sort_values(["predicted_probability", "difficulty"], ascending=[False, True]).copy()
    selected_rows = []
    used_difficulty = 0.0
    used_slots = set()
    for _, row in ordered.iterrows():
        difficulty = row.get("difficulty")
        numeric_difficulty = float(difficulty) if pd.notna(difficulty) else 0.0
        if max_courses is not None and len(selected_rows) >= max_courses:
            break
        if used_difficulty + numeric_difficulty > difficulty_budget:
            continue
        if respect_schedule:
            candidate_slots = set(get_schedule_slots(row))
            if candidate_slots and has_schedule_conflict(list(candidate_slots), used_slots):
                continue
        selected_rows.append(row)
        used_difficulty += numeric_difficulty
        used_slots.update(get_schedule_slots(row))
    if not selected_rows:
        return ordered.head(0).copy()
    return pd.DataFrame(selected_rows).reset_index(drop=True)

# Recommender rules loaded.


---
## Fase 2: Embeddings e Carregamento

O modelo SBERT é carregado e os embeddings do corpus são gerados uma única vez. A busca por similaridade de cosseno executa em menos de 100 ms (~500 MB de RAM). Utiliza-se o pré-treino original do BERTimbau, sem fine-tuning adicional para busca semântica.


In [55]:
# 2.1 Carregar dados, modelo SBERT e gerar embeddings do corpus

df = get_discipline_table()
print(f"Disciplinas: {len(df)}  |  Com ementa detalhada: {sum(~df['ementa'].str.startswith('Estudo dos fundamentos'))}  |  Com horários: {sum(df['horarios'].apply(lambda h: len(h) if isinstance(h, list) else 0) > 0)}")
display(df[['codigo','nome','periodo_sugerido','dificuldade_geral']].head(8))

model = load_sbert_model()
corpus_embeddings = encode_disciplines(model, df)
print(f"Embeddings: {corpus_embeddings.shape}")


Disciplinas: 76  |  Com ementa detalhada: 52  |  Com horários: 41


,codigo,nome,periodo_sugerido,dificuldade_geral
0,DCC217,MATEMÁTICA DISCRETA PARA ENGENHARIA,1.0,5.08
1,ELE630,INTRODUÇÃO A ENGENHARIA DE SISTEMAS,1.0,2.33
2,MAT001,CALCULO DIFERENCIAL E INTEGRAL I,1.0,5.50
3,MAT038,GEOMETRIA ANALITICA E ALGEBRA LINEAR,1.0,6.18
4,QUI628,QUÍMICA GERAL E,1.0,4.39
5,DCC203,PROGRAMAÇÃO E DESENVOLVIMENTO DE SOFTWARE I,2.0,5.07
6,EMT122,INTRODUÇÃO À CIÊNCIA DOS MATERIAIS,2.0,6.33
7,FIS065,FUNDAMENTOS DE MECANICA,2.0,5.42


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embeddings: (76, 768)


---
## Fase 3: Recomendação

A função `recomendar()` constrói o perfil textual, gera o embedding da consulta, ranqueia as disciplinas por similaridade de cosseno (top-20) e aplica os filtros estruturais em sequência: pré-requisitos, balanceamento de dificuldade e conflitos de horário.

Os seis cenários a seguir simulam perfis em diferentes estágios do curso, cada um retornando até 5 disciplinas academicamente viáveis e alinhadas ao interesse declarado.


In [56]:
# 3.1 Função principal de recomendação

def recomendar(interesse, concluidas, *, top_k=20, budget=25.0, max_cursos=5):
    perfil_texto = get_completed_profile_text(df, concluidas)
    query = f"{perfil_texto} {interesse}"

    query_emb = encode_texts(model, [query], batch_size=1)[0]
    hits = semantic_search(query_emb, corpus_embeddings, top_k=top_k)

    candidatas = []
    for hit in hits:
        row = df.iloc[hit['index']].copy()
        row['predicted_probability'] = hit['score']
        row['difficulty'] = row['dificuldade_geral']
        candidatas.append(row)
    df_cand = pd.DataFrame(candidatas)

    df_cand = df_cand[~df_cand['codigo'].isin(concluidas)]

    df_validas = filter_recommended_candidates(df_cand, concluidas)

    df_final = greedy_balance_by_difficulty(df_validas, difficulty_budget=budget, max_courses=max_cursos, respect_schedule=True)

    print(f'Query: "{interesse}"')
    print(f'Concluidas: {concluidas}')
    print(f'Candidatas (top {top_k}): {len(df_cand)}  |  Pós pré-requisitos: {len(df_validas)}  |  Recomendadas: {len(df_final)}')
    print()
    for _, row in df_final.iterrows():
        slots = get_schedule_slots(row)
        horario_str = ', '.join(f'{d} {t}' for d, t in slots) if slots else 'sem horário'
        print(f"  {row['codigo']}: {row['nome']}")
        print(f"    Score: {row['predicted_probability']:.4f}  |  Dificuldade: {row['difficulty']}  |  Horário: {horario_str}")
    return df_final

# recomendar() helper loaded.


---

## Cenário 1: 3º período: computação e software

Concluiu as obrigatórias até o 2º período. Busca disciplinas com foco em computação.


In [57]:
concluidas_s1 = ['MAT001','MAT038','QUI628','ELE630','DCC217', 'MAT039','FIS065','DCC203','EMT122']
recomendar("Tenho muito interesse em programação, software e algoritmos. Gosto da área de computação.", concluidas_s1, budget=22.0, max_cursos=5)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: "Tenho muito interesse em programação, software e algoritmos. Gosto da área de computação."
Concluidas: ['MAT001', 'MAT038', 'QUI628', 'ELE630', 'DCC217', 'MAT039', 'FIS065', 'DCC203', 'EMT122']
Candidatas (top 20): 19  |  Pós pré-requisitos: 17  |  Recomendadas: 5

  ELE055: PLANEJAMENTO DE SISTEMAS DE ENERGIA ELÉTRICA
    Score: 0.8612  |  Dificuldade: nan  |  Horário: sem horário
  EPD117: ORGANIZAÇÃO INDUSTRIAL PARA ENGENHARIA
    Score: 0.8601  |  Dificuldade: nan  |  Horário: sem horário
  ELE059: FUNDAMENTOS DE ENGENHARIA BIOMÉDICA
    Score: 0.8578  |  Dificuldade: 4.17  |  Horário: sem horário
  ENG110: TÓPICOS EM ENGENHARIA DE SISTEMAS II - Introdução ao Controle Automático de Aeronaves (ELT072)
    Score: 0.8575  |  Dificuldade: nan  |  Horário: sem horário
  EEE054: ESTAGIO SUPERVISIONADO EM ENGENHARIA DE SISTEMAS
    Score: 0.8546  |  Dificuldade: 3.2  |  Horário: sem horário


,codigo,nome,carga_horaria,natureza,periodo_sugerido,pre_requisitos,area_conhecimento,ciclo,ementa,dificuldade,trabalhosa,dificuldade_geral,horarios,predicted_probability,difficulty
0,ELE055,PLANEJAMENTO DE SISTEMAS DE ENERGIA ELÉTRICA,45,OB,NaN,[],Engenharia Elétrica e Sistemas,Optativa,Estudo dos fundamentos e aplicações de planeja...,NaN,NaN,NaN,[],0.861187,NaN
1,EPD117,ORGANIZAÇÃO INDUSTRIAL PARA ENGENHARIA,60,OB,NaN,[],Engenharia de Produção,Optativa,Estudo dos fundamentos e aplicações de organiz...,NaN,NaN,NaN,[],0.860091,NaN
2,ELE059,FUNDAMENTOS DE ENGENHARIA BIOMÉDICA,60,OB,NaN,[],Engenharia Elétrica e Sistemas,Optativa,Estudo dos fundamentos e aplicações de fundame...,3.33,5.0,4.17,[],0.857802,4.17
3,ENG110,TÓPICOS EM ENGENHARIA DE SISTEMAS II - Introdu...,30,OB,NaN,[],Engenharia (Tópicos Especiais),Optativa,Estudo de tópicos avançados em tópicos em enge...,NaN,NaN,NaN,[],0.857540,NaN
4,EEE054,ESTAGIO SUPERVISIONADO EM ENGENHARIA DE SISTEMAS,165,OB,11.0,[],Engenharia Elétrica e Sistemas,Integrador,Estágio supervisionado em empresa ou instituiç...,2.20,4.2,3.20,[],0.854645,3.20


---

## Cenário 2: 5º período: inteligência artificial e dados

Já cursou a base de programação e matemática. Busca disciplinas de IA, machine learning e ciência de dados.


In [58]:
concluidas_s2 = [
    'MAT001','MAT038','QUI628','ELE630','DCC217',  # 1º
    'MAT039','FIS065','DCC203','EMT122',             # 2º
    'MAT002','FIS069','DCC204','DCC205','DCC218',    # 3º
    'MAT015','FIS086','FIS152','ELE064','ELT124','EST773',  # 4º
]
recomendar("Meu foco é inteligência artificial, aprendizado de máquina e análise de dados. Quero trabalhar com ciência de dados.", concluidas_s2, budget=22.0, max_cursos=5)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: "Meu foco é inteligência artificial, aprendizado de máquina e análise de dados. Quero trabalhar com ciência de dados."
Concluidas: ['MAT001', 'MAT038', 'QUI628', 'ELE630', 'DCC217', 'MAT039', 'FIS065', 'DCC203', 'EMT122', 'MAT002', 'FIS069', 'DCC204', 'DCC205', 'DCC218', 'MAT015', 'FIS086', 'FIS152', 'ELE064', 'ELT124', 'EST773']
Candidatas (top 20): 18  |  Pós pré-requisitos: 17  |  Recomendadas: 5

  ELE055: PLANEJAMENTO DE SISTEMAS DE ENERGIA ELÉTRICA
    Score: 0.8373  |  Dificuldade: nan  |  Horário: sem horário
  ENG110: TÓPICOS EM ENGENHARIA DE SISTEMAS II - Introdução ao Controle Automático de Aeronaves (ELT072)
    Score: 0.8344  |  Dificuldade: nan  |  Horário: sem horário
  ELE631: ANÁLISE, PROJETO E PROGRAMAÇÃO ORIENTADOS A OBJETOS
    Score: 0.8284  |  Dificuldade: 5.07  |  Horário: TER 19:00-20:40, QUI 20:55-22:35
  ENG121: TÓPICOS EM ENGENHARIA DE SISTEMAS IV - Fundamentos de Redes de Comunicação (ELT036)
    Score: 0.8272  |  Dificuldade: nan  |  Horário: sem hor

,codigo,nome,carga_horaria,natureza,periodo_sugerido,pre_requisitos,area_conhecimento,ciclo,ementa,dificuldade,trabalhosa,dificuldade_geral,horarios,predicted_probability,difficulty
0,ELE055,PLANEJAMENTO DE SISTEMAS DE ENERGIA ELÉTRICA,45,OB,NaN,[],Engenharia Elétrica e Sistemas,Optativa,Estudo dos fundamentos e aplicações de planeja...,NaN,NaN,NaN,[],0.837295,NaN
1,ENG110,TÓPICOS EM ENGENHARIA DE SISTEMAS II - Introdu...,30,OB,NaN,[],Engenharia (Tópicos Especiais),Optativa,Estudo de tópicos avançados em tópicos em enge...,NaN,NaN,NaN,[],0.834367,NaN
2,ELE631,"ANÁLISE, PROJETO E PROGRAMAÇÃO ORIENTADOS A OB...",60,OB,5.0,[DCC204],Engenharia Elétrica e Sistemas,Profissional Engenharia,Gerenciamento da Complexidade; Introdução a An...,4.29,5.86,5.07,"[{'dia': 'TER', 'turno': '19:00-20:40'}, {'dia...",0.828441,5.07
3,ENG121,TÓPICOS EM ENGENHARIA DE SISTEMAS IV - Fundame...,60,OB,NaN,[],Engenharia (Tópicos Especiais),Optativa,Estudo de tópicos avançados em tópicos em enge...,NaN,NaN,NaN,[],0.827188,NaN
4,ELE632,PROCESSOS E MÉTODOS EM ENGENHARIA DE SISTEMAS,60,OB,7.0,[ELE630],Engenharia Elétrica e Sistemas,Profissional Engenharia de Sistemas,Visão holística do sistema socioeconômico atua...,2.56,3.56,3.06,"[{'dia': 'QUI', 'turno': '19:00-20:40'}, {'dia...",0.824583,3.06


---

## Cenário 3: Calouro (1º período): tecnologia e inovação

Sem pré-requisitos cumpridos. Interesse em tecnologia, inovação e empreendedorismo.


In [59]:
concluidas_s3 = []
recomendar("Sou calouro, gosto de tecnologia, inovação e empreendedorismo. Quero matérias que me introduzam à engenharia.", concluidas_s3, budget=20.0, max_cursos=6)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: "Sou calouro, gosto de tecnologia, inovação e empreendedorismo. Quero matérias que me introduzam à engenharia."
Concluidas: []
Candidatas (top 20): 20  |  Pós pré-requisitos: 18  |  Recomendadas: 6

  EMA083: DESENHO MECÂNICO
    Score: 0.7340  |  Dificuldade: nan  |  Horário: sem horário
  EMA092: METROLOGIA
    Score: 0.6981  |  Dificuldade: nan  |  Horário: sem horário
  ELE630: INTRODUÇÃO A ENGENHARIA DE SISTEMAS
    Score: 0.6969  |  Dificuldade: 2.33  |  Horário: SEX 20:55-22:35
  EEE052: TRABALHO DE CONCLUSAO DE CURSO I
    Score: 0.6913  |  Dificuldade: 7.33  |  Horário: TER 19:00-20:40, QUA 19:00-20:40, TER 20:55-22:35
  EEE018: TRABALHO DE CONCLUSAO DE CURSO I
    Score: 0.6913  |  Dificuldade: 7.33  |  Horário: sem horário
  ENG110: TÓPICOS EM ENGENHARIA DE SISTEMAS II - Introdução ao Controle Automático de Aeronaves (ELT072)
    Score: 0.6898  |  Dificuldade: nan  |  Horário: sem horário


,codigo,nome,carga_horaria,natureza,periodo_sugerido,pre_requisitos,area_conhecimento,ciclo,ementa,dificuldade,trabalhosa,dificuldade_geral,horarios,predicted_probability,difficulty
0,EMA083,DESENHO MECÂNICO,45,OB,NaN,[],Engenharia Mecânica/Aeroespacial,Optativa,Estudo dos fundamentos e aplicações de desenho...,NaN,NaN,NaN,[],0.733974,NaN
1,EMA092,METROLOGIA,45,OB,NaN,[],Engenharia Mecânica/Aeroespacial,Optativa,Estudo dos fundamentos e aplicações de metrolo...,NaN,NaN,NaN,[],0.698144,NaN
2,ELE630,INTRODUÇÃO A ENGENHARIA DE SISTEMAS,30,OB,1.0,[],Engenharia Elétrica e Sistemas,Básico Ciências,"Integração do estudante à UFMG, à Escola de En...",1.45,3.21,2.33,"[{'dia': 'SEX', 'turno': '20:55-22:35'}]",0.696863,2.33
3,EEE052,TRABALHO DE CONCLUSAO DE CURSO I,90,OB,10.0,[],Engenharia Elétrica e Sistemas,Integrador,Metodologias científico-tecnológicas e ferrame...,4.67,10.00,7.33,"[{'dia': 'TER', 'turno': '19:00-20:40'}, {'dia...",0.691297,7.33
4,EEE018,TRABALHO DE CONCLUSAO DE CURSO I,90,OB,10.0,[],Engenharia Elétrica e Sistemas,Integrador,Metodologias científico-tecnológicas e ferrame...,4.67,10.00,7.33,[],0.691297,7.33
5,ENG110,TÓPICOS EM ENGENHARIA DE SISTEMAS II - Introdu...,30,OB,NaN,[],Engenharia (Tópicos Especiais),Optativa,Estudo de tópicos avançados em tópicos em enge...,NaN,NaN,NaN,[],0.689787,NaN


---

## Cenário 4: 4º período: optativa em sistemas embarcados

Cursou obrigatórias até o 3º período. Busca optativa em hardware e sistemas embarcados.


In [60]:
concluidas_s4 = [
    'MAT001','MAT038','QUI628','ELE630','DCC217',
    'MAT039','FIS065','DCC203','EMT122',
    'MAT002','FIS069','DCC204','DCC205','DCC218',
]
df_opt = recomendar("Gosto de hardware, circuitos e sistemas embarcados. Preciso de uma optativa nessa área.", concluidas_s4, budget=15.0, max_cursos=2)

optativas = df_opt[df_opt['natureza'] == 'OP'] if len(df_opt) > 0 else df_opt
print(f"\n→ Optativas sugeridas: {len(optativas)}")
for _, row in optativas.iterrows():
    print(f"   {row['codigo']}: {row['nome']} (Score: {row['predicted_probability']:.4f})")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: "Gosto de hardware, circuitos e sistemas embarcados. Preciso de uma optativa nessa área."
Concluidas: ['MAT001', 'MAT038', 'QUI628', 'ELE630', 'DCC217', 'MAT039', 'FIS065', 'DCC203', 'EMT122', 'MAT002', 'FIS069', 'DCC204', 'DCC205', 'DCC218']
Candidatas (top 20): 19  |  Pós pré-requisitos: 17  |  Recomendadas: 2

  ENG110: TÓPICOS EM ENGENHARIA DE SISTEMAS II - Introdução ao Controle Automático de Aeronaves (ELT072)
    Score: 0.8481  |  Dificuldade: nan  |  Horário: sem horário
  ELE055: PLANEJAMENTO DE SISTEMAS DE ENERGIA ELÉTRICA
    Score: 0.8441  |  Dificuldade: nan  |  Horário: sem horário

→ Optativas sugeridas: 0


---

## Cenário 5: Complementar a Fundamentos de IA

Concluiu EEE048 (Fundamentos de IA). Busca disciplinas que aprofundem em machine learning e deep learning.


In [61]:
concluidas_s5 = [
    'MAT001','MAT038','QUI628','ELE630','DCC217','MAT039','FIS065','DCC203','EMT122',
    'MAT002','FIS069','DCC204','DCC205','DCC218','MAT015','FIS086','FIS152','ELE064','ELT124','EST773',
    'ELE065','ELE028','ELT029','ELT084','ELT123','ELE631','ESA019',
    'ELT136','EEE050','ELE082','ELE077',
    'EEE048',  # <- FIA concluída!
]
recomendar("Já fiz Fundamentos de IA. Quero disciplinas que aprofundem em machine learning, deep learning, NLP e visão computacional.", concluidas_s5, budget=20.0, max_cursos=4)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: "Já fiz Fundamentos de IA. Quero disciplinas que aprofundem em machine learning, deep learning, NLP e visão computacional."
Concluidas: ['MAT001', 'MAT038', 'QUI628', 'ELE630', 'DCC217', 'MAT039', 'FIS065', 'DCC203', 'EMT122', 'MAT002', 'FIS069', 'DCC204', 'DCC205', 'DCC218', 'MAT015', 'FIS086', 'FIS152', 'ELE064', 'ELT124', 'EST773', 'ELE065', 'ELE028', 'ELT029', 'ELT084', 'ELT123', 'ELE631', 'ESA019', 'ELT136', 'EEE050', 'ELE082', 'ELE077', 'EEE048']
Candidatas (top 20): 16  |  Pós pré-requisitos: 15  |  Recomendadas: 4

  ELE055: PLANEJAMENTO DE SISTEMAS DE ENERGIA ELÉTRICA
    Score: 0.8275  |  Dificuldade: nan  |  Horário: sem horário
  ENG110: TÓPICOS EM ENGENHARIA DE SISTEMAS II - Introdução ao Controle Automático de Aeronaves (ELT072)
    Score: 0.8258  |  Dificuldade: nan  |  Horário: sem horário
  ENG121: TÓPICOS EM ENGENHARIA DE SISTEMAS IV - Fundamentos de Redes de Comunicação (ELT036)
    Score: 0.8146  |  Dificuldade: nan  |  Horário: sem horário
  ELE632: PROCESSO

,codigo,nome,carga_horaria,natureza,periodo_sugerido,pre_requisitos,area_conhecimento,ciclo,ementa,dificuldade,trabalhosa,dificuldade_geral,horarios,predicted_probability,difficulty
0,ELE055,PLANEJAMENTO DE SISTEMAS DE ENERGIA ELÉTRICA,45,OB,NaN,[],Engenharia Elétrica e Sistemas,Optativa,Estudo dos fundamentos e aplicações de planeja...,NaN,NaN,NaN,[],0.827477,NaN
1,ENG110,TÓPICOS EM ENGENHARIA DE SISTEMAS II - Introdu...,30,OB,NaN,[],Engenharia (Tópicos Especiais),Optativa,Estudo de tópicos avançados em tópicos em enge...,NaN,NaN,NaN,[],0.825788,NaN
2,ENG121,TÓPICOS EM ENGENHARIA DE SISTEMAS IV - Fundame...,60,OB,NaN,[],Engenharia (Tópicos Especiais),Optativa,Estudo de tópicos avançados em tópicos em enge...,NaN,NaN,NaN,[],0.814577,NaN
3,ELE632,PROCESSOS E MÉTODOS EM ENGENHARIA DE SISTEMAS,60,OB,7.0,[ELE630],Engenharia Elétrica e Sistemas,Profissional Engenharia de Sistemas,Visão holística do sistema socioeconômico atua...,2.56,3.56,3.06,"[{'dia': 'QUI', 'turno': '19:00-20:40'}, {'dia...",0.813247,3.06


---

## Cenário 6: 8º período: reta final do curso

Concluiu a maior parte do curso. Precisa das últimas obrigatórias e do TCC, evitando conflitos de horário.


In [62]:
concluidas_s6 = [
    'MAT001','MAT038','QUI628','ELE630','DCC217','MAT039','FIS065','DCC203','EMT122',
    'MAT002','FIS069','DCC204','DCC205','DCC218','MAT015','FIS086','FIS152','ELE064','ELT124','EST773',
    'ELE065','ELE028','ELT029','ELT084','ELT123','ELE631','ESA019',
    'ELT136','EEE050','ELE082','ELE077',
    'ELE632','ELE633','EEE046','EEE017','ELE088',
    'EEE048','EEE049',
]
recomendar("Estou no final do curso. Preciso fechar as matérias que faltam do 8º e 9º períodos.", concluidas_s6, budget=25.0, max_cursos=5)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: "Estou no final do curso. Preciso fechar as matérias que faltam do 8º e 9º períodos."
Concluidas: ['MAT001', 'MAT038', 'QUI628', 'ELE630', 'DCC217', 'MAT039', 'FIS065', 'DCC203', 'EMT122', 'MAT002', 'FIS069', 'DCC204', 'DCC205', 'DCC218', 'MAT015', 'FIS086', 'FIS152', 'ELE064', 'ELT124', 'EST773', 'ELE065', 'ELE028', 'ELT029', 'ELT084', 'ELT123', 'ELE631', 'ESA019', 'ELT136', 'EEE050', 'ELE082', 'ELE077', 'ELE632', 'ELE633', 'EEE046', 'EEE017', 'ELE088', 'EEE048', 'EEE049']
Candidatas (top 20): 15  |  Pós pré-requisitos: 15  |  Recomendadas: 5

  ELE055: PLANEJAMENTO DE SISTEMAS DE ENERGIA ELÉTRICA
    Score: 0.8275  |  Dificuldade: nan  |  Horário: sem horário
  ENG110: TÓPICOS EM ENGENHARIA DE SISTEMAS II - Introdução ao Controle Automático de Aeronaves (ELT072)
    Score: 0.8258  |  Dificuldade: nan  |  Horário: sem horário
  ENG121: TÓPICOS EM ENGENHARIA DE SISTEMAS IV - Fundamentos de Redes de Comunicação (ELT036)
    Score: 0.8146  |  Dificuldade: nan  |  Horário: sem horá

,codigo,nome,carga_horaria,natureza,periodo_sugerido,pre_requisitos,area_conhecimento,ciclo,ementa,dificuldade,trabalhosa,dificuldade_geral,horarios,predicted_probability,difficulty
0,ELE055,PLANEJAMENTO DE SISTEMAS DE ENERGIA ELÉTRICA,45,OB,NaN,[],Engenharia Elétrica e Sistemas,Optativa,Estudo dos fundamentos e aplicações de planeja...,NaN,NaN,NaN,[],0.827477,NaN
1,ENG110,TÓPICOS EM ENGENHARIA DE SISTEMAS II - Introdu...,30,OB,NaN,[],Engenharia (Tópicos Especiais),Optativa,Estudo de tópicos avançados em tópicos em enge...,NaN,NaN,NaN,[],0.825788,NaN
2,ENG121,TÓPICOS EM ENGENHARIA DE SISTEMAS IV - Fundame...,60,OB,NaN,[],Engenharia (Tópicos Especiais),Optativa,Estudo de tópicos avançados em tópicos em enge...,NaN,NaN,NaN,[],0.814577,NaN
3,ELE641,ENGENHARIA DE REABILITAÇÃO E TECNOLOGIAS ASSIS...,60,OB,NaN,[],Engenharia Elétrica e Sistemas,Optativa,Estudo dos fundamentos e aplicações de engenha...,NaN,NaN,NaN,[],0.809979,NaN
4,ELE059,FUNDAMENTOS DE ENGENHARIA BIOMÉDICA,60,OB,NaN,[],Engenharia Elétrica e Sistemas,Optativa,Estudo dos fundamentos e aplicações de fundame...,3.33,5.0,4.17,[],0.809031,4.17


---

## Comparativo entre cenários

Resumo dos filtros estruturais aplicados em cada cenário.


In [63]:
cenarios = {
    "1. 3º período: Software":    ("programação, software, algoritmos", ['MAT001','MAT038','QUI628','ELE630','DCC217','MAT039','FIS065','DCC203','EMT122']),
    "2. 5º período: IA e Dados":  ("inteligência artificial, machine learning, dados", ['MAT001','MAT038','QUI628','ELE630','DCC217','MAT039','FIS065','DCC203','EMT122','MAT002','FIS069','DCC204','DCC205','DCC218','MAT015','FIS086','FIS152','ELE064','ELT124','EST773']),
    "3. Calouro: Tecnologia":     ("tecnologia, inovação, empreendedorismo", []),
    "4. Optativa: Hardware":      ("hardware, circuitos, embarcados", ['MAT001','MAT038','QUI628','ELE630','DCC217','MAT039','FIS065','DCC203','EMT122','MAT002','FIS069','DCC204','DCC205','DCC218']),
    "5. Complementar à FIA":       ("machine learning, deep learning, NLP", ['MAT001','MAT038','QUI628','ELE630','DCC217','MAT039','FIS065','DCC203','EMT122','MAT002','FIS069','DCC204','DCC205','DCC218','MAT015','FIS086','FIS152','ELE064','ELT124','EST773','ELE065','ELE028','ELT029','ELT084','ELT123','ELE631','ESA019','ELT136','EEE050','ELE082','ELE077','EEE048']),
    "6. Avançado: Reta final":    ("fechar matérias finais do curso", ['MAT001','MAT038','QUI628','ELE630','DCC217','MAT039','FIS065','DCC203','EMT122','MAT002','FIS069','DCC204','DCC205','DCC218','MAT015','FIS086','FIS152','ELE064','ELT124','EST773','ELE065','ELE028','ELT029','ELT084','ELT123','ELE631','ESA019','ELT136','EEE050','ELE082','ELE077','ELE632','ELE633','EEE046','EEE017','ELE088','EEE048','EEE049']),
}

resultados = []
for nome, (interesse, concluidas) in cenarios.items():
    perfil = get_completed_profile_text(df, concluidas)
    q = f"{perfil} {interesse}"
    q_emb = encode_texts(model, [q], batch_size=1)[0]
    hits = semantic_search(q_emb, corpus_embeddings, top_k=20)
    cands = []
    for h in hits:
        r = df.iloc[h['index']].copy()
        r['predicted_probability'] = h['score']
        r['difficulty'] = r['dificuldade_geral']
        cands.append(r)
    df_c = pd.DataFrame(cands)
    df_c = df_c[~df_c['codigo'].isin(concluidas)]
    df_v = filter_recommended_candidates(df_c, concluidas)
    df_f = greedy_balance_by_difficulty(df_v, difficulty_budget=22.0, max_courses=5, respect_schedule=True)
    resultados.append({
        "Cenário": nome,
        "Top-20": len(df_c),
        "Pré-req OK": len(df_v),
        "Recomendadas": len(df_f),
        "Top-1": df_f.iloc[0]['codigo'] if len(df_f) > 0 else "—",
        "Top-1 Score": f"{df_f.iloc[0]['predicted_probability']:.3f}" if len(df_f) > 0 else "—",
        "Dificuldade total": f"{df_f['difficulty'].sum():.1f}" if len(df_f) > 0 else "—",
    })

display(pd.DataFrame(resultados).set_index("Cenário"))


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,Top-20,Pré-req OK,Recomendadas,Top-1,Top-1 Score,Dificuldade total
Cenário,,,,,,
1. 3º período: Software,19,17,5,EPD117,0.863,4.2
2. 5º período: IA e Dados,18,17,5,ELE055,0.829,5.1
3. Calouro: Tecnologia,20,15,5,EMA092,0.657,7.3
4. Optativa: Hardware,19,18,5,ELE055,0.838,6.4
5. Complementar à FIA,16,15,5,ELE055,0.827,3.1
6. Avançado: Reta final,15,15,5,ELE055,0.827,4.2


---

## Verificação de conflitos de horário

Comparação da seleção de disciplinas com e sem o filtro `respect_schedule`.


In [64]:
concluidas_sched = ['MAT001','MAT038','QUI628','ELE630','DCC217','MAT039','FIS065','DCC203','EMT122']
perfil = get_completed_profile_text(df, concluidas_sched)
q = f"{perfil} programação, computação, eletrônica"
q_emb = encode_texts(model, [q], batch_size=1)[0]
hits = semantic_search(q_emb, corpus_embeddings, top_k=20)

cands = []
for h in hits:
    r = df.iloc[h['index']].copy()
    r['predicted_probability'] = h['score']
    r['difficulty'] = r['dificuldade_geral']
    cands.append(r)
df_c = pd.DataFrame(cands)
df_c = df_c[~df_c['codigo'].isin(concluidas_sched)]
df_v = filter_recommended_candidates(df_c, concluidas_sched)

print("=== SEM filtro de horário ===")
df_sem = greedy_balance_by_difficulty(df_v, difficulty_budget=22.0, max_courses=5, respect_schedule=False)
for _, row in df_sem.iterrows():
    slots = get_schedule_slots(row)
    h_str = ', '.join(f'{d} {t}' for d,t in slots) if slots else '—'
    print(f"  {row['codigo']}  {h_str}")

all_slots = []
for _, row in df_sem.iterrows():
    all_slots.extend(get_schedule_slots(row))
conflicts = [(s, c) for s, c in Counter(all_slots).items() if c > 1]
print(f"\n  \u26a0 Conflitos detectados: {len(conflicts)}")
for slot, count in conflicts:
    print(f"    {slot[0]} {slot[1]}: {count}x")

print("\n=== COM filtro de horário ===")
df_com = greedy_balance_by_difficulty(df_v, difficulty_budget=22.0, max_courses=5, respect_schedule=True)
for _, row in df_com.iterrows():
    slots = get_schedule_slots(row)
    h_str = ', '.join(f'{d} {t}' for d,t in slots) if slots else '—'
    print(f"  {row['codigo']}  {h_str}")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

=== SEM filtro de horário ===
  EPD117  —
  ELE055  —
  ELE059  —
  EEE054  —
  EEE020  TER 19:00-20:40, QUA 19:00-20:40, QUI 19:00-20:40, TER 20:55-22:35, QUA 20:55-22:35, QUI 20:55-22:35

  ⚠ Conflitos detectados: 0

=== COM filtro de horário ===
  EPD117  —
  ELE055  —
  ELE059  —
  EEE054  —
  EEE020  TER 19:00-20:40, QUA 19:00-20:40, QUI 19:00-20:40, TER 20:55-22:35, QUA 20:55-22:35, QUI 20:55-22:35


---
## Fase 4: Fine-Tuning do Classificador BERT

Classificador binário (1 = relevante, 0 = irrelevante) que recebe pares `[CLS] perfil [SEP] ementa [SEP]` e avalia se a disciplina é adequada ao perfil.

**Transfer Learning progressivo em 3 estágios** (previne overfitting em datasets pequenos):

1. **Feature Extraction** (lr = 5e-5): cabeça de classificação, encoder congelado.
2. **Partial Unfreeze** (lr = 3e-5): descongela as 2 últimas camadas do encoder.
3. **Full Fine-Tuning** (lr = 2e-5): modelo inteiro com learning rate reduzido.

Otimizador AdamW com warmup linear. Os 357 pares de treino são gerados sinteticamente a partir de perfis acumulativos por período, com labels determinísticos baseados nas regras do PPC. Um baseline de Regressão Logística sobre features estruturais serve como sanity check.


In [65]:
# 4.1 Gerador de perfis sintéticos e pares de treino

from dataclasses import dataclass

@dataclass(frozen=True)
class SyntheticProfile:
    profile_id: str
    completed_codes: tuple
    profile_text: str
    variant_name: str

def build_profile_text(disciplines, completed_codes, *, max_tokens_hint=320):
    code_set = {code for code in completed_codes if code}
    selected_rows = disciplines[disciplines["codigo"].isin(code_set)]
    parts = []
    for _, row in selected_rows.iterrows():
        segment_parts = [str(row.get("codigo", "")).strip(), str(row.get("nome", "")).strip()]
        area = row.get("area_conhecimento")
        if pd.notna(area):
            segment_parts.append(f"Area: {area}")
        ementa = str(row.get("ementa", "")).strip()
        if ementa:
            segment_parts.append(ementa[:700])
        parts.append(" | ".join(part for part in segment_parts if part))
    profile_text = "\n\n".join(parts)
    if len(profile_text.split()) > max_tokens_hint * 2:
        return " ".join(profile_text.split()[:max_tokens_hint * 2])
    return profile_text

def generate_period_prefix_profiles(disciplines, *, period_column="periodo_sugerido"):
    profiles = []
    if period_column not in disciplines.columns:
        return profiles
    ordered = disciplines.dropna(subset=[period_column]).sort_values([period_column, "codigo"])
    if ordered.empty:
        return profiles
    max_period = int(ordered[period_column].max())
    for period in range(1, max_period):
        completed_codes = tuple(ordered.loc[ordered[period_column] <= period, "codigo"].dropna().astype(str).tolist())
        profile_text = build_profile_text(disciplines, completed_codes)
        profiles.append(SyntheticProfile(
            profile_id=f"prefix_p{period}",
            completed_codes=completed_codes,
            profile_text=profile_text,
            variant_name="prefix",
        ))
    return profiles

def generate_elective_variants(disciplines, base_profiles, *, group_column="grupo_optativa"):
    variants = []
    if group_column not in disciplines.columns:
        return []
    grouped = disciplines.dropna(subset=[group_column]).groupby(group_column)
    elective_groups = {group_name: group_df["codigo"].dropna().astype(str).tolist() for group_name, group_df in grouped}
    for profile in base_profiles:
        for group_name, group_codes in elective_groups.items():
            chosen = tuple(sorted(set(profile.completed_codes).union(group_codes[:min(2, len(group_codes))])))
            profile_text = build_profile_text(disciplines, chosen)
            variants.append(SyntheticProfile(
                profile_id=f"{profile.profile_id}_{group_name}",
                completed_codes=chosen,
                profile_text=profile_text,
                variant_name=f"elective_{group_name}",
            ))
    return variants

def generate_labelled_pairs(disciplines, profiles):
    records = []
    discipline_codes = disciplines["codigo"].dropna().astype(str).tolist()
    discipline_lookup = disciplines.set_index("codigo")
    for profile in profiles:
        completed_set = set(profile.completed_codes)
        completed_periods = disciplines.loc[disciplines["codigo"].isin(completed_set), "periodo_sugerido"].dropna()
        highest_completed_period = int(completed_periods.max()) if not completed_periods.empty else 0
        completed_groups = set()
        if "grupo_optativa" in disciplines.columns:
            completed_groups = set(
                disciplines.loc[disciplines["codigo"].isin(completed_set), "grupo_optativa"]
                .dropna().astype(str).tolist()
            )
        for code in discipline_codes:
            if code in completed_set:
                continue
            row = discipline_lookup.loc[code]
            prerequisites = set(row.get("pre_requisitos", []) or [])
            prerequisites_met = prerequisites.issubset(completed_set)
            period = row.get("periodo_sugerido")
            same_track = pd.notna(row.get("grupo_optativa")) and str(row.get("grupo_optativa")) in completed_groups
            next_required = pd.notna(period) and int(period) <= highest_completed_period + 1
            positive = bool(prerequisites_met and (same_track or next_required))
            label = int(positive)
            records.append({
                "profile_id": profile.profile_id,
                "variant_name": profile.variant_name,
                "perfil_texto": profile.profile_text,
                "completed_codes": list(profile.completed_codes),
                "candidate_code": code,
                "candidate_text": row.get("ementa", ""),
                "label": label,
                "prerequisites_met": prerequisites_met,
                "same_track": same_track,
                "next_required": next_required,
            })
    return pd.DataFrame(records)

def split_by_profile(pairs, *, train_size=0.7, validation_size=0.15, seed=42):
    unique_profiles = pairs[["profile_id"]].drop_duplicates().sample(frac=1.0, random_state=seed)
    total_profiles = len(unique_profiles)
    if total_profiles < 3:
        return {
            "train": pairs.copy().reset_index(drop=True),
            "validation": pairs.head(0).copy(),
            "test": pairs.head(0).copy(),
        }
    train_cutoff = max(1, int(total_profiles * train_size))
    validation_cutoff = max(train_cutoff + 1, int(total_profiles * (train_size + validation_size)))
    validation_cutoff = min(validation_cutoff, total_profiles - 1)
    train_profiles = set(unique_profiles.iloc[:train_cutoff]["profile_id"])
    validation_profiles = set(unique_profiles.iloc[train_cutoff:validation_cutoff]["profile_id"])
    test_profiles = set(unique_profiles.iloc[validation_cutoff:]["profile_id"])
    return {
        "train": pairs[pairs["profile_id"].isin(train_profiles)].reset_index(drop=True),
        "validation": pairs[pairs["profile_id"].isin(validation_profiles)].reset_index(drop=True),
        "test": pairs[pairs["profile_id"].isin(test_profiles)].reset_index(drop=True),
    }

def generate_sbert_training_pairs(disciplines_df, profiles):
    records = []
    discipline_codes = disciplines_df["codigo"].dropna().astype(str).tolist()
    discipline_lookup = disciplines_df.set_index("codigo")
    for profile in profiles:
        completed_set = set(profile.completed_codes)
        text_a = get_completed_profile_text(disciplines_df, list(completed_set))
        completed_periods = disciplines_df.loc[
            disciplines_df["codigo"].isin(completed_set), "periodo_sugerido"
        ].dropna()
        highest_period = int(completed_periods.max()) if not completed_periods.empty else 0
        for code in discipline_codes:
            if code in completed_set:
                continue
            row = discipline_lookup.loc[code]
            prereqs = set(row.get("pre_requisitos", []) or [])
            prereqs_met = prereqs.issubset(completed_set)
            period = row.get("periodo_sugerido")
            next_required = pd.notna(period) and int(period) <= highest_period + 1
            relevant = bool(prereqs_met and next_required)
            text_b = get_discipline_text(row)
            records.append({
                "text_a": text_a,
                "text_b": text_b,
                "label": 1.0 if relevant else 0.0,
            })
    return pd.DataFrame(records)

# Synthetic profile generator loaded.


In [66]:
# 4.2 Baseline: regressão logística (features estruturais)

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def build_structural_features(pairs):
    data = pairs.copy()
    data["completed_count"] = data["completed_codes"].map(lambda values: len(values) if isinstance(values, list) else 0)
    data["prerequisites_met_int"] = data["prerequisites_met"].astype(int)
    data["same_track_int"] = data["same_track"].astype(int)
    data["next_required_int"] = data["next_required"].astype(int)
    data["candidate_length"] = data["candidate_text"].fillna("").map(lambda text: len(str(text).split()))
    data["profile_length"] = data["perfil_texto"].fillna("").map(lambda text: len(str(text).split()))
    return data[["completed_count", "prerequisites_met_int", "same_track_int", "next_required_int", "candidate_length", "profile_length"]]

def train_baseline(train_pairs, validation_pairs=None):
    feature_builder = ColumnTransformer(
        transformers=[
            ("numeric", Pipeline([("scaler", StandardScaler())]), [
                "completed_count", "prerequisites_met_int", "same_track_int",
                "next_required_int", "candidate_length", "profile_length"
            ]),
        ],
        remainder="drop",
    )
    model = Pipeline(steps=[
        ("features", feature_builder),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ])
    train_features = build_structural_features(train_pairs)
    model.fit(train_features, train_pairs["label"])
    return model

def evaluate_baseline(model, test_pairs):
    test_features = build_structural_features(test_pairs)
    predictions = model.predict(test_features)
    return {
        "accuracy": float(accuracy_score(test_pairs["label"], predictions)),
        "f1": float(f1_score(test_pairs["label"], predictions)),
    }

# Baseline functions loaded.


In [67]:
# 4.3 Classificador BERT com fine-tuning progressivo (3 estágios)

from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup

class PairTextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length_profile=384, max_length_candidate=192):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length_profile = max_length_profile
        self.max_length_candidate = max_length_candidate

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        encoded = self.tokenizer(
            str(row["perfil_texto"]),
            str(row["candidate_text"]),
            truncation=True,
            padding="max_length",
            max_length=max(self.max_length_profile, self.max_length_candidate),
            return_tensors="pt",
        )
        item = {key: value.squeeze(0) for key, value in encoded.items()}
        item["labels"] = torch.tensor(int(row["label"]), dtype=torch.long)
        return item

@dataclass
class TrainingStage:
    name: str
    trainable_layers: str
    epochs: int
    learning_rate: float

def freeze_encoder(model):
    for parameter in model.bert.parameters():
        parameter.requires_grad = False

def unfreeze_last_layers(model, num_layers=2):
    freeze_encoder(model)
    encoder_layers = model.bert.encoder.layer
    for layer in encoder_layers[-num_layers:]:
        for parameter in layer.parameters():
            parameter.requires_grad = True
    for parameter in model.classifier.parameters():
        parameter.requires_grad = True
    if hasattr(model, "dropout"):
        for parameter in model.dropout.parameters():
            parameter.requires_grad = True

def unfreeze_all(model):
    for parameter in model.parameters():
        parameter.requires_grad = True

def create_model(num_labels=2):
    return BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

def create_tokenizer():
    return AutoTokenizer.from_pretrained(MODEL_NAME)

def train_stage(model, data_loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0
    for batch in data_loader:
        batch = {key: value.to(device) for key, value in batch.items()}
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += float(loss.item())
    return total_loss / max(1, len(data_loader))

def predict(model, data_loader, device):
    model.eval()
    predictions = []
    labels = []
    probabilities = []
    with torch.no_grad():
        for batch in data_loader:
            batch = {key: value.to(device) for key, value in batch.items()}
            batch_labels = batch.pop("labels")
            outputs = model(**batch)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1)
            predictions.extend(torch.argmax(probs, dim=-1).cpu().tolist())
            probabilities.extend(probs[:, 1].cpu().tolist())
            labels.extend(batch_labels.cpu().tolist())
    return labels, predictions, probabilities

def evaluate_model(model, data_loader, device):
    labels, predictions, probabilities = predict(model, data_loader, device)
    return {
        "labels": labels,
        "predictions": predictions,
        "probabilities": probabilities,
        "accuracy": float(accuracy_score(labels, predictions)),
        "f1": float(f1_score(labels, predictions)),
    }

def fit_three_stage_model(train_df, validation_df, *, batch_size=8, seed=42):
    torch.manual_seed(seed)
    tokenizer = create_tokenizer()
    model = create_model(num_labels=2)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    training_stages = [
        TrainingStage(name="feature_extraction", trainable_layers="head_only", epochs=1, learning_rate=5e-5),
        TrainingStage(name="partial_unfreeze", trainable_layers="last_two_layers", epochs=1, learning_rate=3e-5),
        TrainingStage(name="full_finetuning", trainable_layers="all", epochs=1, learning_rate=2e-5),
    ]

    history = []
    train_dataset = PairTextDataset(train_df, tokenizer)
    validation_dataset = PairTextDataset(validation_df, tokenizer)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    validation_loader = DataLoader(validation_dataset, batch_size=batch_size)

    for stage in training_stages:
        if stage.trainable_layers == "head_only":
            freeze_encoder(model)
        elif stage.trainable_layers == "last_two_layers":
            unfreeze_last_layers(model, num_layers=2)
        else:
            unfreeze_all(model)

        optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=stage.learning_rate)
        total_steps = max(1, len(train_loader) * stage.epochs)
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=max(1, total_steps // 10), num_training_steps=total_steps)

        for _ in range(stage.epochs):
            loss = train_stage(model, train_loader, optimizer, scheduler, device)
            evaluation = evaluate_model(model, validation_loader, device)
            history.append({"stage": stage.name, "loss": loss, "accuracy": evaluation["accuracy"], "f1": evaluation["f1"]})

    return model, history

# BERT training functions loaded.


In [68]:
# 4.4 Métricas de avaliação (acurácia, precisão, recall, F1)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def classification_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }

def compare_models(results):
    rows = []
    for model_name, metrics in results.items():
        row = {"modelo": model_name, **metrics}
        rows.append(row)
    return pd.DataFrame(rows).set_index("modelo")

# Evaluation functions loaded.


In [69]:
# 4.5 Execução: gerar dados, treinar modelos, comparar resultados

print("Gerando perfis sintéticos...")
base_profiles = generate_period_prefix_profiles(df)
all_profiles = base_profiles + generate_elective_variants(df, base_profiles)
pairs = generate_labelled_pairs(df, all_profiles)
splits = split_by_profile(pairs, seed=42)

print(f"Perfis sintéticos: {len(all_profiles)}")
print(f"Pares de treino:   {len(splits['train'])} (train)  |  {len(splits['validation'])} (val)  |  {len(splits['test'])} (test)")
print(f"Distribuição de labels (train): {splits['train']['label'].value_counts().to_dict()}")

print("\nTreinando baseline estrutural...")
baseline = train_baseline(splits['train'])
baseline_metrics = evaluate_baseline(baseline, splits['test'])
print(f"Baseline (Logistic Regression) → Accuracy: {baseline_metrics['accuracy']:.3f}  |  F1: {baseline_metrics['f1']:.3f}")

print("\nIniciando fine-tuning do BERT (3 estágios)...")
bert_model, history = fit_three_stage_model(splits['train'], splits['validation'], batch_size=4, seed=42)

print("\nHistórico de treino:")
for entry in history:
    print(f"  {entry['stage']:22s} → loss: {entry['loss']:.4f}  acc: {entry['accuracy']:.3f}  f1: {entry['f1']:.3f}")

print("\nAvaliando BERT no conjunto de teste...")
tokenizer = create_tokenizer()
test_dataset = PairTextDataset(splits['test'], tokenizer)
test_loader = DataLoader(test_dataset, batch_size=4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model.to(device)

y_true, y_pred, _ = predict(bert_model, test_loader, device)
bert_metrics = classification_metrics(y_true, y_pred)

comparison = compare_models({
    "Baseline (Log. Reg.)": baseline_metrics,
    "BERT (fine-tuned)": bert_metrics,
})
print("\nComparação no conjunto de teste:")
display(comparison)


Gerando perfis sintéticos...
Perfis sintéticos: 10
Pares de treino:   357 (train)  |  51 (val)  |  100 (test)
Distribuição de labels (train): {0: 334, 1: 23}

Treinando baseline estrutural...
Baseline (Logistic Regression) → Accuracy: 1.000  |  F1: 1.000

Iniciando fine-tuning do BERT (3 estágios)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th


Histórico de treino:
  feature_extraction     → loss: 0.7305  acc: 0.765  f1: 0.000
  partial_unfreeze       → loss: 0.2745  acc: 0.902  f1: 0.000
  full_finetuning        → loss: 0.2228  acc: 0.902  f1: 0.000

Avaliando BERT no conjunto de teste...

Comparação no conjunto de teste:


,accuracy,f1,precision,recall
modelo,,,,
Baseline (Log. Reg.),1.00,1.0,NaN,NaN
BERT (fine-tuned),0.89,0.0,0.0,0.0
